# Three-Model Evaluation

Compare Transformer, LSTM, and FCN checkpoints on the same test groups and plot the shared ground truth once per group.

## Load libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from pathlib import Path
from tqdm import tqdm

from shaft_force_sensing.training.utils import load_model
from shaft_force_sensing.evaluation import tb_to_numpy, array_bais, array_medfilt

%load_ext autoreload
%autoreload 2

## Configure checkpoints

In [ ]:
CKPT_ROOT = Path("../logs")

CHECKPOINT_DIRS = {
    "transformer": CKPT_ROOT / "transformer" / "base",
    "lstm": CKPT_ROOT / "lstm" / "base",
    "fcn": CKPT_ROOT / "fcn" / "base",
}

MODEL_ORDER = ["transformer", "lstm", "fcn"]
PLOT_COLORS = {
    "transformer": "#1f77b4",
    "lstm": "#ff7f0e",
    "fcn": "#2ca02c",
}

models = {}
for name in MODEL_ORDER:
    models[name] = load_model(CHECKPOINT_DIRS[name], map_location="cpu")

In [ ]:
teleop_flags = {name: bool(model.hparams.get("teleop", False)) for name, model in models.items()}
if len(set(teleop_flags.values())) != 1:
    raise ValueError(f"All models must use the same dataset split. Got: {teleop_flags}")

teleop = next(iter(teleop_flags.values()))
test_split_name = "Teleop" if teleop else "Automated"

test_roots = {name: CHECKPOINT_DIRS[name] / test_split_name for name in MODEL_ORDER}
common_groups = ['Free_0003', 'Palpation_0006', 'Traction_0003']

## Helpers

In [ ]:
axes = ["F_x", "F_y", "F_z"]

lims = [
    (100, 140),
    (100, 140),
    (100, 140),
]

if len(common_groups) != len(lims):
    raise ValueError(f"Expected {len(lims)} test groups, but got {len(common_groups)}: {common_groups}")

GROUP_LIMS = {group: lim for group, lim in zip(common_groups, lims)}
PLOT_LABELS = {
    "transformer": "Transformer",
    "lstm": "LSTM",
    "fcn": "FCN",
}
DRAW_ORDER = ["fcn", "lstm", "transformer"]
LEGEND_ORDER = ["Transformer", "LSTM", "FCN", "GT"]


def postprocess_prediction(pred: np.ndarray, model_name: str) -> np.ndarray:
    pred = array_medfilt(pred)

    if model_name != "lstm" and teleop:
        pred = array_bais(pred)

    return pred


def load_group_data(group_name: str) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    gt_reference = None
    predictions = {}

    for model_name in MODEL_ORDER:
        group_path = test_roots[model_name] / group_name
        gt, pred = tb_to_numpy(group_path)
        pred = postprocess_prediction(pred, model_name)

        if gt_reference is None:
            gt_reference = gt
        elif not np.allclose(gt_reference, gt):
            raise ValueError(f"Ground truth mismatch for {group_name} in {model_name}.")

        predictions[model_name] = pred

    return gt_reference, predictions


def preload_group_data(group_names: list[str]) -> dict[str, tuple[np.ndarray, dict[str, np.ndarray]]]:
    cached = {}
    for group_name in tqdm(group_names, desc="Loading group data"):
        cached[group_name] = load_group_data(group_name)
    return cached


def plot_group_grid(
    group_names: list[str],
    group_data: dict[str, tuple[np.ndarray, dict[str, np.ndarray]]],
    sampling_rate: float = 100.0,
) -> Figure:
    n_rows = len(axes)
    n_cols_left = len(group_names)
    n_cols_total = n_cols_left + 1

    width_ratios = [1.0] * n_cols_left + [1.0]
    fig, axes_grid = plt.subplots(
        n_rows,
        n_cols_total,
        figsize=(4 * n_cols_left + 4, 2 * n_rows),
        squeeze=False,
        gridspec_kw={"width_ratios": width_ratios},
    )

    # Share x-axis only within left time-series columns; keep scatter column independent.
    for col_idx in range(n_cols_left):
        for row_idx in range(1, n_rows):
            axes_grid[row_idx, col_idx].sharex(axes_grid[0, col_idx])

    for col_idx, group_name in enumerate(group_names):
        if group_name not in group_data:
            raise KeyError(f"Missing preloaded data for group: {group_name}")

        gt, predictions = group_data[group_name]
        xlim = GROUP_LIMS.get(group_name)

        x_gt = np.arange(gt.shape[0], dtype=float) / sampling_rate

        for row_idx, axis_name in enumerate(axes):
            ax = axes_grid[row_idx, col_idx]
            y_window_values = []
            y_gt = gt[:, row_idx]

            ax.plot(x_gt, y_gt, color="black", linewidth=2, label="GT")

            if xlim is not None:
                gt_mask = (x_gt >= xlim[0]) & (x_gt <= xlim[1])
                y_window_values.append(y_gt[gt_mask] if np.any(gt_mask) else y_gt)
            else:
                y_window_values.append(y_gt)

            for model_name in DRAW_ORDER:
                pred = predictions[model_name]
                x_pred = np.arange(pred.shape[0], dtype=float) / sampling_rate
                y_pred = pred[:, row_idx]
                ax.plot(
                    x_pred,
                    y_pred,
                    color=PLOT_COLORS[model_name],
                    linewidth=1.6,
                    label=PLOT_LABELS[model_name],
                )

                if xlim is not None:
                    pred_mask = (x_pred >= xlim[0]) & (x_pred <= xlim[1])
                    y_window_values.append(y_pred[pred_mask] if np.any(pred_mask) else y_pred)
                else:
                    y_window_values.append(y_pred)

            y_concat = np.concatenate(y_window_values)
            y_concat = y_concat[np.isfinite(y_concat)]
            if y_concat.size > 0:
                y_min = np.floor(np.min(y_concat))
                y_max = np.ceil(np.max(y_concat))
                if y_min == y_max:
                    y_min -= 1.0
                    y_max += 1.0
                ax.set_ylim(y_min, y_max)
                ax.set_yticks([y_min, y_max])

            if row_idx == 0:
                ax.set_title(group_name)

            if col_idx == 0:
                ax.set_ylabel(f'${axis_name}$ (N)')
            else:
                ax.set_ylabel("")

            if xlim is not None:
                ax.set_xlim(xlim)
                x_min, x_max = xlim
            else:
                x_min, x_max = ax.get_xlim()

            ax.set_xticks([x_min, x_max])
            ax.set_xticklabels([f"{x_min:g}", f"{x_max:g}"])

            ax.grid(False)

    # Add a rightmost scatter column aligned by row with the time-series grid.
    scatter_col_idx = n_cols_total - 1
    for row_idx, axis_name in enumerate(axes):
        ax_scatter = axes_grid[row_idx, scatter_col_idx]
        axis_values = []

        for model_name in DRAW_ORDER:
            y_true_parts = []
            y_pred_parts = []

            for group_name in group_names:
                if group_name not in group_data:
                    raise KeyError(f"Missing preloaded data for group: {group_name}")

                gt, predictions = group_data[group_name]
                pred = predictions[model_name]
                if gt.shape != pred.shape:
                    raise ValueError(
                        f"Shape mismatch in group '{group_name}' for model '{model_name}': gt={gt.shape}, pred={pred.shape}"
                    )

                y_true_parts.append(gt[:, row_idx])
                y_pred_parts.append(pred[:, row_idx])

            y_true = np.concatenate(y_true_parts, axis=0)
            y_pred = np.concatenate(y_pred_parts, axis=0)
            axis_values.append(y_true)
            axis_values.append(y_pred)

            ax_scatter.scatter(
                y_true,
                y_pred,
                s=5,
                alpha=0.45,
                color=PLOT_COLORS[model_name],
                edgecolor="none",
                rasterized=True,
                label=PLOT_LABELS[model_name] if row_idx == 0 else None,
            )

        axis_concat = np.concatenate(axis_values)
        axis_concat = axis_concat[np.isfinite(axis_concat)]
        min_val = np.floor(np.min(axis_concat))
        max_val = np.ceil(np.max(axis_concat))
        if min_val == max_val:
            min_val -= 1.0
            max_val += 1.0

        ax_scatter.plot([min_val, max_val], [min_val, max_val], "k--", lw=0.8)
        ax_scatter.set_xlim(min_val, max_val)
        ax_scatter.set_ylim(min_val, max_val)
        ax_scatter.set_xticks([min_val, max_val])
        ax_scatter.set_yticks([min_val, max_val])
        ax_scatter.set_xticklabels([int(min_val), int(max_val)])
        ax_scatter.set_yticklabels([int(min_val), int(max_val)])
        ax_scatter.set_aspect("equal", "box")
        ax_scatter.set_title("Scatter" if row_idx == 0 else "")
        ax_scatter.grid(False)

        if row_idx == n_rows - 1:
            ax_scatter.set_xlabel("True Force (N)")

    fig.supxlabel("Time (s)", y=0.08, x=0.42)

    handles, labels = axes_grid[0, 0].get_legend_handles_labels()
    handle_map = {label: handle for handle, label in zip(handles, labels)}
    ordered_labels = [label for label in LEGEND_ORDER if label in handle_map]
    ordered_handles = [handle_map[label] for label in ordered_labels]
    fig.legend(
        ordered_handles,
        ordered_labels,
        loc="lower center",
        ncol=len(ordered_labels),
        frameon=False,
        bbox_to_anchor=(0.38, 0.0),
    )
    fig.tight_layout(rect=[0, 0.14, 1, 0.95])

    return fig

In [ ]:
# Run this cell once. Re-run only when checkpoints/data settings change.
cached_group_data = preload_group_data(common_groups)

In [ ]:
grid_fig = plot_group_grid(common_groups, cached_group_data, sampling_rate=100.0)

output_pdf = Path("../logs/results/automated_time_plot.pdf")
output_pdf.parent.mkdir(parents=True, exist_ok=True)
grid_fig.savefig(output_pdf, format="pdf", bbox_inches="tight")

plt.show()
plt.close(grid_fig)
print(f"Saved: {output_pdf.resolve()}")